# 💬 기능 1. 논문 AI 에이전트 채팅 플랫폼 (기초)

본 가이드는 **Paper Agent** 프로젝트의 **대화형 서비스 (기능 1)** 중 **LLM API를 활용한 답변 생성 및 구조화 출력(Structured Output)의 핵심 동작 원리**를 친절하게 다룹니다.

데이터베이스 CRUD나 세션 관리 같은 단순 DB 데이터 조작보다는, **실제 대화 답변을 생성하고 제목을 작명하는 AI 모델의 핵심 구동 흐름**을 단계별로 실습하며 이해해 봅니다.

---

## 📌 기초 학습 핵심 포인트

1. **LLM 모델 객체 초기화** (`init_chat_model`)
   - LangChain 통합 인터페이스를 사용해 OpenAI의 `gpt-4o-mini` 모델을 동적으로 적재하는 기법을 알아봅니다.
2. **구조화된 답변(Structured Output) 추출**
   - Pydantic 스키마 모델을 LLM에 주입하여, 생성된 답변 텍스트와 근거 논문 정보가 정확한 JSON 규격으로 반환되는 구조를 살펴봅니다.
3. **대화방 제목 자동 생성**
   - 사용자가 질문한 첫 마디에 대해 AI가 맥락을 이해하고 간결한 한국어 방 제목(6~20자)을 스스로 작명하는 로직을 시뮬레이션합니다.

In [8]:
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

# Pydantic v2와 LangChain 연동 시 내부 직렬화(to_python) 과정에서 발생하는 무해한 UserWarning 무시
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

# 현재 notebooks/chat_service/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env


## 2단계. LangChain Structured Output의 필요성

일반적인 AI 채팅 기능은 단순 문자열(String)만 반환합니다. 하지만 우리의 논문 추천 시스템에서는 다음과 같이 구조화된 데이터가 함께 반환되어야 합니다:
- AI가 질문에 답하는 **자연스러운 설명 (explanation)**
- 설명의 근거가 된 **논문 목록 (papers)** (각 논문의 `title`, `arxiv_id`, `summary` 포함)

이를 일반 텍스트에서 정규표현식 등으로 추출하려면 에러가 잦고 번거롭습니다. LangChain의 `with_structured_output()` 기능을 활용하면, Pydantic 모델을 바인딩하여 LLM이 직접 약속된 JSON 포맷(정확한 타입 보장)으로 응답을 리턴하도록 강제할 수 있습니다.

In [ ]:
import asyncio
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# 1. Pydantic DTO 스키마 정의 (테스트 목적)
class PaperRef(BaseModel):
    title: str = Field(description="논문 제목")
    arxiv_id: str = Field(description="논문의 arXiv ID")
    summary: str = Field(description="논문에 대한 한 줄 요약")

class StructuredAnswer(BaseModel):
    explanation: str = Field(description="사용자 질문에 대한 핵심 설명")
    papers: list[PaperRef] = Field(default_factory=list, description="답변의 근거가 된 논문 목록")

async def run_llm_demo():
    # 2. LLM 모델 객체 초기화 (gpt-4o-mini)
    llm = init_chat_model("openai:gpt-4o-mini", temperature=0.2)
    
    # 3. plain_llm 호출 테스트
    print("🤖 [1. Plain Text 호출] 일반 텍스트 형태로 답변을 생성합니다...")
    plain_response = await llm.ainvoke("Describe Kepler's laws of planetary motion briefly in Korean.")
    print("📝 답변:\n", plain_response.content)
    print("\n" + "="*50 + "\n")
    
    # 4. 구조화 출력 모델(Structured Output) 바인딩
    print("🤖 [2. Structured Output 호출] Pydantic 스키마를 통해 구조화된 JSON 답변을 생성합니다...")
    structured_llm = llm.with_structured_output(StructuredAnswer)
    
    prompt = (
        "소행성체 형성에 대해 한국어로 설명하고, 이에 관한 임의의 논문 2편을 근거 논문(papers) 리스트로 합성해줘.\n"
        "- 설명은 자연스러운 한국어로 작성하고,\n"
        "- 논문 목록은 적절한 arxiv_id와 영어 제목을 사용해서 채워줘."
    )
    
    structured_response = await structured_llm.ainvoke(prompt)
    if not isinstance(structured_response, StructuredAnswer):
        raise TypeError(f"Expected StructuredAnswer, got {type(structured_response)}")
    
    # 5. 파싱 결과 확인
    print("✅ Pydantic 객체로 즉시 반환 완료!")
    print(f"📄 설명: {structured_response.explanation}")
    print("📚 인용된 논문 목록:")
    for idx, paper in enumerate(structured_response.papers, 1):
        print(f"  [{idx}] Title: {paper.title}")
        print(f"      arXiv ID: {paper.arxiv_id}")
        print(f"      Summary: {paper.summary}")

await run_llm_demo()

## 3단계. 첫 질문에 따른 대화방 제목(Title) 자동 작명 원리

채팅 목록 화면을 깔끔하게 유지하기 위해, 첫 대화가 입력되었을 때 질문 내용을 분석하여 한국어의 핵심 키워드 위주로 간결한 제목(6~20자)을 자동 생성합니다.

이 로직은 데이터베이스 저장이나 도구 사용이 불필요한 단발성 텍스트 변환이므로, 무거운 Agent 대신 **경량화된 LLM 단일 모델(`gpt-4o-mini`)을 직접 호출**하여 가볍고 빠르게 처리합니다.

In [ ]:
async def run_title_generation_demo():
    llm = init_chat_model("openai:gpt-4o-mini", temperature=0.7)
    
    test_questions = [
        "유방암 표적 치료를 위한 CRISPR-Cas9 유전자 편집 기법의 최신 동향을 가르쳐주세요.",
        "Planetesimal formation and planetary migration in the early solar system.",
        "딥러닝을 활용한 이미지 초해상도(Super Resolution) 알고리즘 성능 비교 논문 검색해줘."
    ]
    
    print("🚀 [Title Generation] 테스트 질문들로부터 제목 작명을 개시합니다...\n")
    for idx, q in enumerate(test_questions, 1):
        prompt = (
            "다음 질문을 한국어로 6~20자의 간결한 채팅방 제목으로 만들어줘.\n"
            "- 제목만 출력하고 따옴표나 군더더기 설명은 붙이지 마.\n"
            "- 질문의 핵심 주제를 자연스럽게 요약해.\n\n"
            f"질문: {q}"
        )
        
        response = await llm.ainvoke(prompt)
        content = response.content
        if isinstance(content, str):
            title = content.strip().strip('"').strip("'")
        else:
            title = ""
        
        print(f"❓ 원본 질문 {idx}: \"{q}\"")
        print(f"✨ 생성된 제목: \"{title}\"\n")

await run_title_generation_demo()

## 4단계. 백엔드 구현 파일 연계 정보

실제 백엔드 소스코드에서 이 LLM 호출 기법들은 아래와 같이 구현되어 연동됩니다:

1. **`ChatAgent.run()` / `ChatAgent.run_stream()`**
   - [chat_agent.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/chat/chat_agent.py) 파일 내에 정의되어 있으며, 여기서 본 가이드와 유사하게 `gpt-4o-mini` 모델과 `BioAnswer` (Pydantic) 모델을 이용하여 구조화 출력을 생성합니다.
2. **`ChatAgent.generate_title()`**
   - [chat_agent.py:L345](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/chat/chat_agent.py#L345)에 정의되어 있으며, 사용자의 첫 번째 질문을 기반으로 간결한 방 제목을 지어주는 기능을 담당합니다.
3. **`ChatService` 및 REST API 라우터**
   - [services.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/chat/services.py)와 [endpoints.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/chat/endpoints.py)를 통해 사용자의 대화 요청을 Agent로 흘려보내고, 생성된 결과 및 출처를 DB와 클라이언트로 연결해 줍니다.